# ElasticSearchのベクトル検索のシンプルなサンプル

ElasticSearchクライアントを使用するサンプル。

※ElasticSearchおよびクライアントのバージョン：8.18.0

ただし、今後、ElasticSearchの最新版を使う場合、ElasticSearchクライアントの対応まで時間がかかることがあり、また、互換性がなく修正が必要になる可能性がある。

## 必要パッケージのインポート

In [ ]:
from elasticsearch8 import Elasticsearch
from sentence_transformers import SentenceTransformer
import time

## 設定

In [ ]:
ES_URL = 'http://llm-rag-examples-elasticsearch1:9200'
INDEX_NAME = 'vector_test02'

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
MODEL_DIM = 384

## 埋め込みモデル初期化

In [ ]:
model = SentenceTransformer(MODEL_NAME)

## ElasticSearchオブジェクト生成

In [ ]:
es = Elasticsearch(
    [ES_URL],
    headers={
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
)

## インデックスの有無チェック、なければエラーで終了

In [ ]:
if not es.indices.exists(index=INDEX_NAME):
    raise Exception(f'index {INDEX_NAME} がありません。まず、登録処理を実行してください。（elasticsearch-vector-ex02-01-es-client-insert.ipynb）')

## ベクトル検索

In [ ]:
QUERY_TEXT = '日本の都市'

In [ ]:
query_vector = model.encode(QUERY_TEXT)

In [ ]:
search_body = {
    'size': 3,
    'knn': {
        'field': 'vector',
        'query_vector': query_vector.tolist(),
        'k': 3,
        'num_candidates': 100
    }
}

In [ ]:
search_result = es.search(index=INDEX_NAME, body=search_body)

In [ ]:
search_result

In [ ]:
# --- 検索結果表示 ---
for hit in search_result['hits']['hits']:
    print(f'スコア: {hit['_score']:.4f} テキスト: {hit['_source']['text']}')